How did SpaceX’s booster landing results change between 2010 and 2018, and do mission type (orbit, payload mass) relate to landing success?

## Project Scope

**Goal:** Find out whether SpaceX's booster landings improved between 2010 and 2018,
and whether orbit or payload mass relates to landing success.



**Data:** spacex_launch_data.csv (Kaggle) - 57 launches, 11 columns.
Key columns: Date, Landing Outcome, Orbit, Payload Mass (kg).
Limits: small sample, and only 39 launches attempted a landing.



**Analysis steps:**
1. Clean the data (fix dates, tidy Landing Outcome labels, make Payload Mass numeric, group orbits)
2. Launches per year
3. Landing success rate by year (attempts only)
4. Compare outcomes by orbit and payload mass



**Evaluation:** Do the totals still add up to 57 after cleaning? Does each result
answer the question, and are the conclusions honest about the small sample?



**Output:** A notebook with charts and written insights, plus a README on GitHub.


**Audience:** "Curious general readers interested in space"

In [1]:
import pandas as pd


In [2]:
launches = pd.read_csv("Data/spacex_launch_data.csv")

In [3]:
print(launches.shape)

(57, 11)


In [4]:
launches.info()

<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Flight Number      57 non-null     str  
 1   Date               57 non-null     str  
 2   Time (UTC)         57 non-null     str  
 3   Booster Version    57 non-null     str  
 4   Launch Site        57 non-null     str  
 5   Payload            57 non-null     str  
 6   Payload Mass (kg)  55 non-null     str  
 7   Orbit              57 non-null     str  
 8   Customer           57 non-null     str  
 9   Mission Outcome    57 non-null     str  
 10  Landing Outcome    57 non-null     str  
dtypes: str(11)
memory usage: 11.2 KB


In [5]:
launches.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Mission Outcome,Landing Outcome
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute)
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",NaN,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525,LEO (ISS),NASA (COTS),Success,No attempt
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [6]:
print(launches["Landing Outcome"].value_counts())
print(launches["Orbit"].value_counts())
print(launches["Mission Outcome"].value_counts())
launches["Date"].min(), launches["Date"].max()

Landing Outcome
No attempt                                                          18
Success (drone ship)                                                11
Success (ground pad)                                                 8
Controlled (ocean)                                                   5
Failure  (drone ship)                                                3
Success  (drone ship)                                                3
Failure   (parachute)                                                2
Uncontrolled (ocean)                                                 2
Failure (drone ship)                                                 2
Precluded  (drone ship)                                              1
Success  (ground pad)                                                1
Success (ground pad), Failure (drone ship), Success (ground pad)     1
Name: count, dtype: int64
Orbit
GTO                                                          22
LEO (ISS)                           

('2010-06-04', '2018-06-04')

In [7]:
print(launches["Landing Outcome"].value_counts())

Landing Outcome
No attempt                                                          18
Success (drone ship)                                                11
Success (ground pad)                                                 8
Controlled (ocean)                                                   5
Failure  (drone ship)                                                3
Success  (drone ship)                                                3
Failure   (parachute)                                                2
Uncontrolled (ocean)                                                 2
Failure (drone ship)                                                 2
Precluded  (drone ship)                                              1
Success  (ground pad)                                                1
Success (ground pad), Failure (drone ship), Success (ground pad)     1
Name: count, dtype: int64


In [8]:
launches["Flight Number"].nunique()

57

In [9]:
launches[["Date", "Landing Outcome", "Orbit", "Payload Mass (kg)"]].isna().sum()

Date                 0
Landing Outcome      0
Orbit                0
Payload Mass (kg)    2
dtype: int64

In [10]:
attempts = launches[launches["Landing Outcome"] != "No attempt"]

In [11]:
print(len(attempts))

39


In [12]:
launches["Date"].str[:4].value_counts().sort_index()

Date
2010     2
2012     2
2013     3
2014     6
2015     7
2016     8
2017    18
2018    11
Name: count, dtype: int64

## Data check summary

- **One row = one SpaceX launch** (57 launches, June 2010 to June 2018).
  
- **Columns used:** "Date", "Landing Outcome", "Orbit" and "Payload Mass (kg)".

- **Missing data:** "Payload Mass (kg)" is missing for 2 launches. "Date", "Landing Outcome" and "Orbit" have no gaps.

- **Is there enough data?** The launches are not evenly spread across years. 2010 to 2013 have only 2 or 3 launches a year, there are none in 2011, 2018 is a partial year (to June), and only 39 launches attempted a landing. Results for single years may be unreliable, so I will group years or show counts next to any rates.
  

## Data Wrangling and Tidying.

In [13]:
clean = launches.copy()

In [14]:
clean.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Mission Outcome,Landing Outcome
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute)
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",NaN,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525,LEO (ISS),NASA (COTS),Success,No attempt
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [15]:
clean['Date'] = pd.to_datetime(clean["Date"])

In [16]:
print(clean.dtypes)

Flight Number                   str
Date                 datetime64[us]
Time (UTC)                      str
Booster Version                 str
Launch Site                     str
Payload                         str
Payload Mass (kg)               str
Orbit                           str
Customer                        str
Mission Outcome                 str
Landing Outcome                 str
dtype: object


In [17]:
clean["Payload Mass (kg)"] = pd.to_numeric(clean["Payload Mass (kg)"], errors = "coerce")

In [18]:
clean["Payload Mass (kg)"].isna().sum()

np.int64(57)

In [19]:
clean["Payload Mass (kg)"] = launches["Payload Mass (kg)"]

In [20]:
print(launches["Payload Mass (kg)"].tolist())

[nan, nan, '525\xa0', '500\xa0', '677\xa0', '500\xa0', '3,170\xa0', '3,325\xa0', '2,296\xa0', '1,316\xa0', '4,535\xa0', '4,428\xa0', '2,216\xa0', '2,395\xa0', '570\xa0', '4,159\xa0', '1,898\xa0', '4,707\xa0', '1,952\xa0', '2,034\xa0', '553\xa0', '5,271\xa0', '3,136\xa0', '4,696\xa0', '3,100\xa0', '3,600\xa0', '2,257\xa0', '4,600\xa0', '9,600\xa0', '2,490\xa0', '5,600\xa0', '5,300\xa0', 'Classified', '6,070\xa0', '2,708\xa0', '3,669\xa0', '9,600\xa0', '6,761\xa0', '3,310\xa0', '475\xa0', '4,990\xa0', '9,600\xa0', '5,200\xa0', '3,500\xa0', '2,205\xa0', '9,600\xa0', 'Classified', '4,230\xa0', '~1,250\xa0', '2,150\xa0', '6,092\xa0', '9,600\xa0', '2,647\xa0', '362\xa0', '3,600\xa0', '6,460\xa0', '5,384\xa0']


In [21]:
clean["Payload Mass (kg)"] = (
    launches["Payload Mass (kg)"]
    .str.replace("\xa0", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("~", "", regex=False)
)
clean["Payload Mass (kg)"] = pd.to_numeric(clean["Payload Mass (kg)"], errors="coerce")

In [22]:
clean["Payload Mass (kg)"].isna().sum()

np.int64(4)

In [23]:
print(clean.dtypes)

Flight Number                   str
Date                 datetime64[us]
Time (UTC)                      str
Booster Version                 str
Launch Site                     str
Payload                         str
Payload Mass (kg)           float64
Orbit                           str
Customer                        str
Mission Outcome                 str
Landing Outcome                 str
dtype: object


In [24]:
clean["Landing Outcome"] = launches["Landing Outcome"].str.split().str.join(" ")


In [25]:
print(clean["Landing Outcome"].value_counts())

Landing Outcome
No attempt                                                          18
Success (drone ship)                                                14
Success (ground pad)                                                 9
Controlled (ocean)                                                   5
Failure (drone ship)                                                 5
Failure (parachute)                                                  2
Uncontrolled (ocean)                                                 2
Precluded (drone ship)                                               1
Success (ground pad), Failure (drone ship), Success (ground pad)     1
Name: count, dtype: int64


In [26]:
clean["Year"] = clean["Date"].dt.year

In [27]:
#print(clean["Year"].value_counts().sort_index())

In [28]:
result_map = {
    "No attempt": "No attempt",
    "Success (drone ship)": "Success",
    "Success (ground pad)": "Success",
    "Failure (drone ship)": "Failure",
    "Failure (parachute)": "Failure",
    "Controlled (ocean)": "Other",
    "Uncontrolled (ocean)": "Other",
    "Precluded (drone ship)": "Other",
    "Success (ground pad), Failure (drone ship), Success (ground pad)": "Other",
}

clean["Landing Result"] = clean["Landing Outcome"].map(result_map)
print(clean["Landing Result"].value_counts(dropna=False))

Landing Result
Success       23
No attempt    18
Other          9
Failure        7
Name: count, dtype: int64


In [30]:
clean.head(60)

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Mission Outcome,Landing Outcome,Year,Landing Result
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute),2010,Failure
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",NaN,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),2010,Failure
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),Success,No attempt,2012,No attempt
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),Success,No attempt,2012,No attempt
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),Success,No attempt,2013,No attempt
5,6,2013-09-29,16:00:00,F9 v1.1 B1003,VAFB SLC-4E,CASSIOPE,500.0,Polar LEO,MDA,Success,Uncontrolled (ocean),2013,Other
6,7,2013-12-03,22:41:00,F9 v1.1,CCAFS LC-40,SES-8,3170.0,GTO,SES,Success,No attempt,2013,No attempt
7,8,2014-01-06,22:06:00,F9 v1.1,CCAFS LC-40,Thaicom 6,3325.0,GTO,Thaicom,Success,No attempt,2014,No attempt
8,9,2014-04-18,19:25:00,F9 v1.1,CCAFS LC-40,SpaceX CRS-3,2296.0,LEO (ISS),NASA (CRS),Success,Controlled (ocean),2014,Other
9,10,2014-07-14,15:15:00,F9 v1.1,CCAFS LC-40,OG2 Mission 1 6 Orbcomm-OG2 satellites,1316.0,LEO,Orbcomm,Success,Controlled (ocean),2014,Other


In [39]:
clean["Orbit"] = clean["Orbit"].str.split().str.join(" ")

In [40]:
clean["Orbit"].value_counts()

Orbit
GTO                                                         22
LEO (ISS)                                                   16
LEO                                                          8
Polar LEO                                                    6
SSO                                                          2
Sun–Earth L1                                                 1
Heliocentric 0.99–1.67 AU (close to Mars transfer orbit)     1
HEO                                                          1
Name: count, dtype: int64

In [65]:
orbit_map = { "GTO": "GTO", 
              "LEO (ISS)": "LEO", 
              "LEO": "LEO", 
              "Polar LEO": "LEO", 
              "SSO": "LEO",
              "Sun–Earth L1": "Other", 
              "Heliocentric 0.99–1.67 AU (close to Mars transfer orbit)": "Other", 
              "HEO": "Other"}

In [66]:
clean["Orbit_Group"] = clean["Orbit"].map(orbit_map)

In [67]:
print(clean["Orbit_Group"].value_counts(dropna = False))

Orbit_Group
LEO      32
GTO      22
Other     3
Name: count, dtype: int64


In [68]:
print(clean[clean["Orbit_Group"].isna()]["Orbit"].unique())

[]


In [73]:
clean.shape

(57, 14)

In [74]:
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Flight Number      57 non-null     str           
 1   Date               57 non-null     datetime64[us]
 2   Time (UTC)         57 non-null     str           
 3   Booster Version    57 non-null     str           
 4   Launch Site        57 non-null     str           
 5   Payload            57 non-null     str           
 6   Payload Mass (kg)  53 non-null     float64       
 7   Orbit              57 non-null     object        
 8   Customer           57 non-null     str           
 9   Mission Outcome    57 non-null     str           
 10  Landing Outcome    57 non-null     object        
 11  Year               57 non-null     int32         
 12  Landing Result     57 non-null     str           
 13  Orbit_Group        57 non-null     str           
dtypes: datetime64[us](1), f